# Recommendation Systems - Complete Usage Examples

This notebook demonstrates how to use Content-Based, Collaborative Filtering, and Hybrid recommendation models.

In [ ]:
import pandas as pd
import numpy as np
from src.data_processing.data_loader import MovieLensDataLoader
from src.models.content_based import ContentBasedRecommender, ContentBasedConfig
from src.models.collaborative_filtering import CollaborativeFiltering
from src.models.hybrid import HybridRecommender
import warnings
warnings.filterwarnings('ignore')

## 1. Load and Prepare Data

In [ ]:
loader = MovieLensDataLoader()
data_dict = loader.load_data()
await loader.letterboxd_data_async(max_concurrent_requests=500)

movies_df = pd.DataFrame(loader.movie_data)
ratings_df = data_dict['ratings']
genre_features = loader.preprocess_movies()
movies_df = pd.concat([movies_df, genre_features], axis=1)
movies_df = movies_df.dropna().reset_index(drop=True)

print(f"Movies: {len(movies_df)}")
print(f"Ratings: {len(ratings_df)}")
print(f"Users: {ratings_df['userId'].nunique()}")

## 2. Content-Based Filtering

In [ ]:
config = ContentBasedConfig(
    main_actor_weight=0.3,
    director_weight=0.3,
    cast_weight=0.3,
    keywords_weight=0.6,
    numerical_weight=0.1,
    similarity_threshold=0.15,
    top_k_default=10
)

cb_model = ContentBasedRecommender(config=config)
cb_model.fit(movies_df=movies_df, ratings_df=ratings_df)
print("Content-Based model trained successfully!")

### 2.1 Get Recommendations for a User

In [ ]:
user_id = 3
user_history = ratings_df[ratings_df['userId'] == user_id]
watched_items = set(user_history['movieId'].values)
liked_items = set(user_history[user_history['rating'] >= 4.0]['movieId'].values)

print(f"User {user_id} has watched {len(watched_items)} movies")
print(f"User {user_id} liked {len(liked_items)} movies (rating >= 4.0)")

In [ ]:
recommendations = cb_model.get_top_k_with_titles(
    user_id=user_id,
    watched_items=watched_items,
    k=10
)

print(f"\nTop 10 Recommendations for User {user_id}:")
print("=" * 80)
for i, rec in enumerate(recommendations, 1):
    print(f"{i:2}. {rec['title']:<50} (Score: {rec['score']:.4f})")

### 2.2 Explain Recommendations

In [ ]:
movie_id_to_title = dict(zip(movies_df['movieId'], movies_df['title']))

if recommendations:
    rec_movie_id = recommendations[0]['movieId']
    rec_title = recommendations[0]['title']
    
    print(f"Why was '{rec_title}' recommended?")
    print("=" * 80)
    
    reasons = cb_model.explain_recommendation(
        movie_id=rec_movie_id,
        liked_items=liked_items,
        top_n_reasons=5
    )
    
    for i, reason in enumerate(reasons, 1):
        reason_title = movie_id_to_title.get(reason['movie_id'], f"ID: {reason['movie_id']}")
        print(f"{i}. {reason_title} (Similarity: {reason['similarity']:.4f})")

### 2.3 Find Similar Movies

In [ ]:
target_movie_id = 1.0
target_movie_title = movies_df[movies_df['movieId'] == target_movie_id]['title'].values[0]

print(f"Movies similar to '{target_movie_title}':")
print("=" * 80)

similar_movies = cb_model.get_similar_movies(
    movie_id=target_movie_id,
    top_n=10
)

for i, movie in enumerate(similar_movies, 1):
    title = movie_id_to_title.get(movie['movie_id'], f"ID: {movie['movie_id']}")
    print(f"{i:2}. {title:<50} (Similarity: {movie['similarity']:.4f})")

### 2.4 Predict Scores for Specific Movies

In [ ]:
candidate_movies = [1.0, 2.0, 3.0, 4.0, 5.0]
scores = cb_model.predict_scores(user_id=user_id, item_ids=candidate_movies)

print(f"Predicted scores for User {user_id}:")
print("=" * 80)
for movie_id, score in zip(candidate_movies, scores):
    title = movie_id_to_title.get(movie_id, f"ID: {movie_id}")
    print(f"{title:<50} -> Score: {score:.4f}")

## 3. Collaborative Filtering

In [ ]:
cf_model = CollaborativeFiltering(
    k_components=50,
    random_state=42
)
cf_model.fit(df_ratings=ratings_df)
print("Collaborative Filtering model trained successfully!")

### 3.1 Get Recommendations

In [ ]:
cf_recommendations = cf_model.get_top_k_recommendations(
    user_id=user_id,
    watched_items=watched_items,
    k=10
)

print(f"\nTop 10 CF Recommendations for User {user_id}:")
print("=" * 80)
for i, movie_id in enumerate(cf_recommendations, 1):
    title = movie_id_to_title.get(movie_id, f"ID: {movie_id}")
    score = cf_model.predict_score(user_id, movie_id)
    print(f"{i:2}. {title:<50} (Predicted Rating: {score:.4f})")

### 3.2 Explain CF Recommendations

In [ ]:
if cf_recommendations:
    cf_movie_id = cf_recommendations[0]
    cf_title = movie_id_to_title.get(cf_movie_id, f"ID: {cf_movie_id}")
    
    print(f"Why was '{cf_title}' recommended (CF)?")
    print("=" * 80)
    
    cf_reasons = cf_model.explain_recommendation(
        movie_id=cf_movie_id,
        liked_items=liked_items,
        top_n_reasons=5
    )
    
    for i, reason in enumerate(cf_reasons, 1):
        reason_title = movie_id_to_title.get(reason['movie_id'], f"ID: {reason['movie_id']}")
        print(f"{i}. {reason_title} (Similarity: {reason['similarity']:.4f})")

## 4. Hybrid Model

In [ ]:
hybrid_model = HybridRecommender(
    cf_model=cf_model,
    cb_model=cb_model,
    alpha=0.5
)

hybrid_model.fit(movies_df=movies_df, ratings_df=ratings_df)
print("Hybrid model trained successfully!")

### 4.1 Get Hybrid Recommendations

In [ ]:
hybrid_recommendations = hybrid_model.get_top_k_recommendations(
    user_id=user_id,
    watched_items=watched_items,
    k=10
)

print(f"\nTop 10 Hybrid Recommendations for User {user_id} (alpha=0.5):")
print("=" * 80)
for i, movie_id in enumerate(hybrid_recommendations, 1):
    title = movie_id_to_title.get(movie_id, f"ID: {movie_id}")
    score = hybrid_model.predict_scores(user_id, [movie_id])[0]
    print(f"{i:2}. {title:<50} (Score: {score:.4f})")

### 4.2 Explain Hybrid Recommendations

In [ ]:
if hybrid_recommendations:
    hybrid_movie_id = hybrid_recommendations[0]
    hybrid_title = movie_id_to_title.get(hybrid_movie_id, f"ID: {hybrid_movie_id}")
    
    print(f"Why was '{hybrid_title}' recommended (Hybrid)?")
    print("=" * 80)
    
    hybrid_reasons = hybrid_model.explain_recommendation(
        movie_id=hybrid_movie_id,
        liked_items=liked_items,
        top_n_reasons=5
    )
    
    for i, reason in enumerate(hybrid_reasons, 1):
        reason_title = movie_id_to_title.get(reason['movie_id'], f"ID: {reason['movie_id']}")
        print(f"{i}. {reason_title} (Combined Similarity: {reason['similarity']:.4f})")

## 5. Compare All Three Models

In [ ]:
print(f"\nComparison of Top 5 Recommendations for User {user_id}:")
print("=" * 120)
print(f"{'Rank':<6}{'Content-Based':<35}{'Collaborative':<35}{'Hybrid':<35}")
print("-" * 120)

for i in range(5):
    cb_title = movie_id_to_title.get(recommendations[i]['movieId'], "N/A")[:33] if i < len(recommendations) else "N/A"
    cf_title = movie_id_to_title.get(cf_recommendations[i], "N/A")[:33] if i < len(cf_recommendations) else "N/A"
    hybrid_title = movie_id_to_title.get(hybrid_recommendations[i], "N/A")[:33] if i < len(hybrid_recommendations) else "N/A"
    
    print(f"{i+1:<6}{cb_title:<35}{cf_title:<35}{hybrid_title:<35}")

## 6. Save and Load Models

In [ ]:
cb_model.save_artifacts(
    similarity_path="data/processed/artifacts/cb_similarity.npy",
    mapping_path="data/processed/artifacts/cb_mapping.pkl",
    preprocessors_path="data/processed/artifacts/cb_preprocessors.pkl"
)
print("Content-Based model saved!")

In [ ]:
cb_model_loaded = ContentBasedRecommender(config=config)
cb_model_loaded.load_artifacts(
    similarity_path="data/processed/artifacts/cb_similarity.npy",
    mapping_path="data/processed/artifacts/cb_mapping.pkl",
    preprocessors_path="data/processed/artifacts/cb_preprocessors.pkl"
)
print("Content-Based model loaded!")

test_recommendations = cb_model_loaded.get_top_k_recommendations(
    user_id=user_id,
    watched_items=watched_items,
    k=5
)
print(f"Test: Generated {len(test_recommendations)} recommendations with loaded model")

## 7. Full User Profile Display

In [ ]:
cb_model.show_user_profile_and_recommendations(
    user_id=user_id,
    ratings_df=ratings_df,
    movies_df=movies_df,
    k=10,
    top_rated_count=5,
    reasons_count=3
)